# Session 06 Topic 03: The 95% confidence interval and its width

Use this notebook while working through Topic 03.

This notebook builds a **95% confidence interval**, then examines the three things that
control its width.

A confidence interval is the most useful thing you can hand a decision-maker, and also
the most frequently misread — so a good part of this notebook is about what the phrase
"95% confident" does and does not license you to say.

Charts and simulations are supplied and commented. You will write the interval
calculations and answer the activity questions.

## 1. Setup

The same filter as the previous two notebooks: Sandringham, normal weekdays, 2023-24.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")

DATA_FOLDER = Path("data")
TRAIN_FILE = DATA_FOLDER / "train_daily_boardings.csv"

train = pd.read_csv(TRAIN_FILE, parse_dates=["business_date"])

sandringham = train[
    (train["line_name"] == "Sandringham")
    & (train["day_type"] == "Normal Weekday")
    & (train["financial_year"] == "2023-24")
]

boardings = sandringham["total_boardings"]

print("Days:", len(boardings))
print("Mean:", round(boardings.mean()))

## 2. Building the interval

Every confidence interval has the same two parts:

$$\text{interval} = \text{point estimate} \pm \text{margin of error}$$

The point estimate is the sample mean. The margin of error is the standard error
scaled up by a **critical value** that sets the confidence level:

$$E = t_c \times \frac{s}{\sqrt{n}}$$

For 95% confidence $t_c$ is close to 1.96 — the number from Topic 02. It is not exactly
1.96 because with a sample we must estimate the standard deviation as well as the mean,
and the **t-distribution** compensates by being slightly wider than the normal curve.
The smaller the sample, the bigger the compensation.

Build the four ingredients first.

In [ ]:
# Store the sample size as n and the mean as mean.
# Calculate the standard error: the standard deviation divided by
#   the square root of n.
# Get the critical value with stats.t.ppf(0.975, n - 1).
# Multiply it by the standard error to get the margin of error.
# Print all five values.

The interval is then the mean plus and minus that margin. `scipy` will do the whole
thing in one line, and you should check that the two agree.

In [ ]:
# Print the interval worked out by hand: mean minus and plus the margin.
# Then use stats.t.interval(0.95, df=n-1, loc=mean, scale=standard_error)
#   and print that too. The two should match.

You should have **[40,707, 42,213]**.

Now compare the margin of error with the spread of the data itself. These are very
different numbers, and confusing them is the commonest mistake in the whole topic.

In [ ]:
print("Standard deviation of daily boardings:", round(boardings.std()))
print("Margin of error for the mean:         ", round(margin_of_error))

Individual days swing widely. The *average* of 202 of them is tightly pinned down to
within about 750. That contrast is the benefit of averaging, and it is why the interval
is so much narrower than the data.

### Activity — Build one yourself


Calculate the 95% confidence interval for **Williamstown** normal weekdays in 2023-24.
Report n, the mean, and both bounds, then write the result as a sentence a manager
would understand.

**Your answer:** Double-click this cell and replace this text with your response.

In [ ]:
# Filter train to Williamstown, Normal Weekday, 2023-24 and take total_boardings.
# Work out n, the mean and the standard error, as you did above.
# Use stats.t.interval to get the bounds, and print everything.
# Also print the standard deviation - you will need it for the answer.

## 3. What it means

The correct reading of the Sandringham result is:

> We are 95% confident that the mean number of weekday boardings on the Sandringham
> line lies between 40,707 and 42,213.

The confidence is a statement about **the method**, not about this particular interval.
That is exactly what you demonstrated at the end of Topic 02: 95% of intervals built
this way capture the true mean. Your one interval either contains it or it does not —
you cannot tell which.

## 4. Three things it does not mean

**"95% of daily boardings fall between 40,707 and 42,213."**

This is the big one. The interval is about the **mean**, not about individual days. Run
the cell below to see how wrong it is.

In [ ]:
# Where do the middle 95% of individual DAYS actually fall?
day_low = boardings.quantile(0.025)
day_high = boardings.quantile(0.975)

print("Interval for the MEAN:      [", round(low), ",", round(high), "]  width", round(high - low))
print("Middle 95% of actual DAYS:  [", round(day_low), ",", round(day_high), "]  width", round(day_high - day_low))
print()
print("Days below the interval's lower bound:", (boardings < low).sum(), "out of", n)
print("Days above the interval's upper bound:", (boardings > high).sum(), "out of", n)

Most days fall **outside** the confidence interval. That is not a contradiction — the
interval was never about individual days.

**"There is a 95% probability the population mean is in this interval."**

Tempting, and almost right, but the population mean is a fixed number. It is either
inside your interval or outside it. The 95% describes how often the procedure succeeds
across many samples, not the odds for this one.

**"A wider interval means the data is worse."**

Not necessarily. A wide interval honestly reports genuine variability. A narrow
interval from a biased sample is far more dangerous, because it is confidently wrong.

### Activity — Diagnose the error


Each statement below misreads the interval [40,707, 42,213]. Name the mistake in each.

1. "Almost every weekday sees between 40,707 and 42,213 boardings."
2. "There is a 5% chance the true average is above 42,213."
3. "The interval is narrow, so the data must be accurate."

**Your answer:** Double-click this cell and replace this text with your response.

## 5. What makes an interval wide or narrow

Three levers control the width, and only one of them is really under your control.

### Sample size

More data narrows the interval, because $n$ sits under a square root in the standard
error. This is the lever you can pull, and it has steeply diminishing returns.

In [ ]:
# Store the standard deviation of boardings as spread.
# Loop over the sample sizes 25, 50, 100 and 200.
# For each, get the critical value, then work out the full width:
#   2 * critical * spread / sqrt(size).
# Print the size and the width.

In [ ]:
# The same relationship across every sample size from 10 to 200.
sizes = range(10, 201, 5)
widths = []

for size in sizes:
    critical = stats.t.ppf(0.975, size - 1)
    widths.append(2 * critical * spread / np.sqrt(size))

plt.figure(figsize=(8, 4.5))
plt.plot(list(sizes), widths, linewidth=2.5, color="#0072B2")

# Mark the two sizes from the table above.
for size in [25, 100]:
    critical = stats.t.ppf(0.975, size - 1)
    width = 2 * critical * spread / np.sqrt(size)
    plt.plot(size, width, "o", color="#D55E00", markersize=9)

plt.title("Width falls steeply at first, then flattens")
plt.xlabel("Sample size (number of days)")
plt.ylabel("Width of the 95% interval")
plt.show()

Quadrupling the sample halves the width — 25 days give about 4,481 and 100 days about
2,154. Halving it again would take 400 days, and again 1,600.

Past a point, more data stops being worth the cost. That is a useful thing to be able
to tell a client who wants more precision.

### Spread in the data

A more variable population gives a wider interval at the same sample size. You cannot
fix this by collecting more carefully — it is a property of the thing you are
measuring. That is why Williamstown's interval came out so much narrower than
Sandringham's.

Sometimes wide spread is a signal that the group is really two groups. Splitting them
narrows both intervals because each group is more homogeneous than the combined group.

### Confidence level

Demanding more confidence forces a wider interval. There is no way round the trade-off.

In [ ]:
# Loop over the confidence levels 0.90, 0.95 and 0.99.
# For each, use stats.t.interval to get the bounds.
# Print the level, both bounds, and the width.

A 100% confidence interval would run from zero to infinity — certain, and completely
useless. 95% is the conventional compromise, and this unit uses it throughout.

### Activity — Choose the lever


A client says your interval is too wide to act on. For each response, say whether it
would work and what it would cost.

1. Collect another two years of data.
2. Report a 90% interval instead of 95%.
3. Analyse Mondays separately from the rest of the week.

**Your answer:** Double-click this cell and replace this text with your response.

## 6. A small sample, for contrast

Take just 30 of the 202 days. The machinery is identical; the answer is much less
useful.

In [ ]:
# Take a sample of 30 days, with replace=True and random_state=1106.
# Work out its n, mean and standard error.
# Build its 95% interval with stats.t.interval.
# Print it next to the interval from all 202 days, with both widths.

The 30-day interval is about 4,200 wide against 1,500 from the full 202 days.

Both are honest. The second is simply more informative — and, importantly, **the
interval tells you that itself**. A reader given only the two means would have no way
to know that one was three times shakier than the other.

## What you have done

- Built a 95% confidence interval by hand and with `stats.t.interval()`, and checked
  they agree
- Seen that the interval for the **mean** is about fourteen times narrower than the
  spread of individual days, and that most days fall outside it
- Practised the correct reading, and diagnosed three misreadings
- Measured how width responds to **sample size**, **spread** and **confidence level**
- Found that separating groups can be a better move than collecting more data

**Next:** Topic 04's notebook puts several intervals side by side and uses them to
compare groups — the point of the whole session.